# 类继承关系
```mermaid
classDiagram
    class Records
    class Ranges
    class Drawdowns
    
    %% 继承关系
    Records <|-- Ranges
    Ranges <|-- Drawdowns
```

# 回撤

## 数学定义
对于一个价格序列
$${x_1},{x_2}, \cdots ,{x_n}$$
$x_p$ 是**峰值**当且仅当
$${x_p} \ge {x_{p - 1}},{x_p} \ge {x_{p + 1}}$$
- 序列端点需特殊处理，${x_1}$ 是峰值如果 ${x_1} \ge {x_2}$，${x_n}$ 是峰值如果 ${x_n} \ge {x_{n-1}}$
- 如果存在连续峰值 ${x_p}$ 和 ${x_{p+1}}$（必有${x_p} = {x_{p + 1}}$），则只考虑后面的 ${x_{p+1}}$

对于一个峰值 ${x_p}$，其对应的**恢复点**：$p$ 之后第一个满足 ${x_r} \ge {x_p}$ 的 $r$。

峰值 ${x_p}$ 和其恢复点 ${x_r}$ 的**谷值**定义为
$${x_v} = \min \left\{ {{x_t}|t = p,p + 1, \cdots ,r} \right\}$$
一次**回撤事件**就是一组 $\left( {p,v,r} \right)$，**回撤幅度**定义为
$$drawdown = \frac{{{x_v} - {x_p}}}{{{x_p}}}$$

## 数据结构
参考 [records/decorators.ipynb](../records/decorators.ipynb) 中的 *关于例子的说明*：
- `drawdown_dt`：表示 *回撤* 的数据结构
- `dd_field_config`：数据结构映射，包括每个字段的名称 `name`，显示标题 `title`，以及映射或取值范围 `mapping`
- `dd_attach_field_config`：每个字段是否生成属性/可过滤的属性（根据取值）

```python
drawdown_dt = np.dtype([
    ('id', np.int64),           # 记录唯一标识符：回撤事件的唯一ID
    ('col', np.int64),          # 列索引：标识属于哪一列数据（资产/策略）
    ('peak_idx', np.int64),     # 峰值索引：回撤开始前的最高点时间索引
    ('start_idx', np.int64),    # 开始索引：回撤正式开始的时间索引（峰值后第一个下跌点）
    ('valley_idx', np.int64),   # 谷值索引：回撤过程中的最低点时间索引
    ('end_idx', np.int64),      # 结束索引：回撤结束的时间索引（恢复到峰值或序列结束）
    ('peak_val', np.float64),   # 峰值价格：回撤开始前的最高价格值
    ('valley_val', np.float64), # 谷值价格：回撤过程中的最低价格值
    ('end_val', np.float64),    # 结束价格：回撤结束时的价格值
    ('status', np.int64),       # 状态标识：使用DrawdownStatus枚举值
], align=True)                  # 内存对齐优化，确保高效的数据访问
"""
```

```python
dd_field_config = Config(
    dict(
        # 指定回撤记录使用的数据类型，包含回撤分析所需的所有字段
        dtype=drawdown_dt,
        
        # 字段设置：定义每个字段的显示标题和映射关系
        settings=dict(
            # 回撤ID字段：用于唯一标识每个回撤记录
            id=dict(
                title='Drawdown Id'            # 字段显示标题
            ),
            
            # 峰值索引字段：标识回撤开始时的峰值位置
            peak_idx=dict(
                title='Peak Timestamp',        # 字段显示标题：峰值时间戳
                mapping='index'               # 映射到ArrayWrapper的index属性
            ),
            
            # 谷底索引字段：标识回撤的最低点位置
            valley_idx=dict(
                title='Valley Timestamp',      # 字段显示标题：谷底时间戳
                mapping='index'               # 映射到ArrayWrapper的index属性
            ),
            
            # 峰值价格字段：记录回撤开始时的价格水平
            peak_val=dict(
                title='Peak Value',           # 字段显示标题：峰值价格
            ),
            
            # 谷底价格字段：记录回撤的最低价格水平
            valley_val=dict(
                title='Valley Value',         # 字段显示标题：谷底价格
            ),
            
            # 结束价格字段：记录回撤结束时的价格水平
            end_val=dict(
                title='End Value',            # 字段显示标题：结束价格
            ),
            
            # 回撤状态字段：标识回撤是否已恢复
            status=dict(
                mapping=DrawdownStatus        # 映射到DrawdownStatus枚举类
            )
        )
    ),
    readonly=True,      # 配置为只读，防止运行时修改
    as_attrs=False     # 不将配置项作为属性访问
)
```

```python
dd_attach_field_config = Config(
    dict(
        # 为status字段启用过滤器功能
        status=dict(
            attach_filters=True        # 自动生成按状态过滤的方法，如filter_by_status
        )
    ),
    readonly=True,    # 配置为只读
    as_attrs=False   # 不将配置项作为属性访问
)
```

## 模块 `class Drawdowns(Ranges)`
`Ranges` 子类，专门用于处理回撤记录。
```python
@attach_fields(dd_attach_field_config)
@override_field_config(dd_field_config)
class Drawdowns(Ranges):
```

### `__init__`

```python
def __init__(self,
                wrapper: ArrayWrapper,
                records_arr: tp.RecordArray,
                ts: tp.Optional[tp.ArrayLike] = None,
                **kwargs) -> None:
    Ranges.__init__(
        self,
        wrapper,
        records_arr,
        ts=ts,
        **kwargs
    )
    self._ts = ts
```

### drawdown
计算每个回撤的回撤幅度。

```python
@cached_property
def drawdown(self) -> MappedArray:

    drawdown = nb.dd_drawdown_nb(
        self.get_field_arr('peak_val'),
        self.get_field_arr('valley_val')
    )
    return self.map_array(drawdown)
```

#### 例子

In [3]:
import vectorbt as vbt
import pandas as pd

# 创建价格数据
price = pd.Series([100, 105, 98, 95, 102, 108, 103])
drawdowns = vbt.Drawdowns.from_ts(price)

# 获取回撤幅度
dd_values = drawdowns.drawdown
print("回撤幅度:")
print(dd_values.values)

回撤幅度:
[-0.0952381 -0.0462963]


### active_drawdown

```python
@cached_method
def active_drawdown(self, group_by: tp.GroupByLike = None,
                    wrap_kwargs: tp.KwargsLike = None) -> tp.MaybeSeries:
    """Drawdown of the last active drawdown only.

    Does not support grouping."""
    if self.wrapper.grouper.is_grouped(group_by=group_by):
        raise ValueError("Grouping is not supported by this method")
    wrap_kwargs = merge_dicts(dict(name_or_index='active_drawdown'), wrap_kwargs)
    active = self.active
    curr_end_val = active.end_val.nth(-1, group_by=group_by)
    curr_peak_val = active.peak_val.nth(-1, group_by=group_by)
    curr_drawdown = (curr_end_val - curr_peak_val) / curr_peak_val
    return self.wrapper.wrap_reduced(curr_drawdown, group_by=group_by, **wrap_kwargs)
```

#### 活跃回撤

恢复点 $r$ 是序列的最后一个时间点 $n$，并且 ${x_n} < {x_p}$ 。

#### 例子

In [4]:
import vectorbt as vbt
import pandas as pd

# 创建包含活跃回撤的价格数据
price = pd.Series([100, 105, 98, 95, 97])  # 最后没有回到峰值
drawdowns = vbt.Drawdowns.from_ts(price)

# 获取活跃回撤幅度
active_dd = drawdowns.active_drawdown()
print(f"当前活跃回撤: {active_dd:.2%}")

# 风险监控示例
if active_dd is not None and active_dd < -0.1:
    print("警告：当前回撤超过10%")

# 多资产监控
portfolio = pd.DataFrame({
    'Stock_A': [100, 105, 98, 95, 97],
    'Stock_B': [100, 110, 90, 85, 88],
    'Stock_C': [100, 102, 101, 103, 104]  # 无活跃回撤
})

portfolio_dd = vbt.Drawdowns.from_ts(portfolio)

print("各资产活跃回撤:")
for col in portfolio.columns:
    try:
        active_dd = portfolio_dd[col].active_drawdown()
        if pd.notna(active_dd):
            print(f"{col}: {active_dd:.2%}")
        else:
            print(f"{col}: 无活跃回撤")
    except:
        print(f"{col}: 无活跃回撤")

# 实时风险管理
risk_threshold = -0.15
for col in portfolio.columns:
    try:
        active_dd = portfolio_dd[col].active_drawdown()
        if pd.notna(active_dd) and active_dd < risk_threshold:
            print(f"风险警告：{col}的活跃回撤({active_dd:.2%})超过阈值({risk_threshold:.2%})")
    except:
        pass

当前活跃回撤: -7.62%
各资产活跃回撤:
Stock_A: -7.62%
Stock_B: -20.00%
Stock_C: 无活跃回撤
风险警告：Stock_B的活跃回撤(-20.00%)超过阈值(-15.00%)
